In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

file_path = "/content/drive/MyDrive/crop recommendation 45/Copy of crop_recommendation_45_with_year_area.csv"

df_tune = pd.read_csv(file_path)

print("Dataset shape:", df_tune.shape)
print("Has crop_year:", "crop_year" in df_tune.columns)
print("Has area:", "area" in df_tune.columns)

print("\nColumns:")
print(df_tune.columns.tolist())

Dataset shape: (218189, 14)
Has crop_year: True
Has area: True

Columns:
['state_names', 'district_names', 'season_names', 'soil_for_crop', 'crop_names', 'N', 'P', 'K', 'temperature', 'humidity', 'rainfall_mm', 'npk_source', 'crop_year', 'area']


In [ ]:

# STEP 5 — Prepare correct Area + Year features

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer

categorical_features = [
    "state_names",
    "district_names",
    "season_names",
    "soil_for_crop"
]

numeric_features = [
    "N",
    "P",
    "K",
    "temperature",
    "humidity",
    "rainfall_mm",
    "crop_year",
    "area"
]

all_features = categorical_features + numeric_features

X = df_tune[all_features]
y_raw = df_tune["crop_names"]

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

# Same random split as the original experiment
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_features
        ),
        ("num", "passthrough", numeric_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Train:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)
print("Crop classes:", len(label_encoder.classes_))


Train: (152732, 673)
Validation: (32728, 673)
Test: (32729, 673)
Crop classes: 45


In [ ]:

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

estimator_results = {}

for n in [800, 900, 1000]:
    print(f"\nTraining with n_estimators={n}...")

    model = XGBClassifier(
        n_estimators=n,
        max_depth=8,
        learning_rate=0.07,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=1,
        gamma=0,
        objective="multi:softprob",
        num_class=len(label_encoder.classes_),
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_processed,
        y_train,
        eval_set=[(X_val_processed, y_val)],
        verbose=False
    )

    predictions = model.predict(X_test_processed)
    accuracy = accuracy_score(y_test, predictions)
    estimator_results[n] = accuracy

    print(f"Test accuracy: {accuracy * 100:.2f}%")

print("\n--- N_ESTIMATORS COMPARISON ---")
for n, accuracy in estimator_results.items():
    print(f"n_estimators={n}: {accuracy * 100:.2f}%")

best_n = max(estimator_results, key=estimator_results.get)
print(f"\nBest n_estimators: {best_n}")
print(f"Best accuracy: {estimator_results[best_n] * 100:.2f}%")



Training with n_estimators=300...
Test accuracy: 94.12%

Training with n_estimators=500...
Test accuracy: 94.42%

Training with n_estimators=700...
Test accuracy: 94.48%

--- N_ESTIMATORS COMPARISON ---
n_estimators=300: 94.12%
n_estimators=500: 94.42%
n_estimators=700: 94.48%

Best n_estimators: 700
Best accuracy: 94.48%


In [ ]:

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

lambda_results = {}

for lam in [1, 3, 5]:
    print(f"\nTraining with reg_lambda={lam}...")

    model = XGBClassifier(
        n_estimators=700,
        max_depth=8,
        learning_rate=0.07,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=1,
        gamma=0,
        reg_lambda=lam,
        objective="multi:softprob",
        num_class=len(label_encoder.classes_),
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_processed,
        y_train,
        eval_set=[(X_val_processed, y_val)],
        verbose=False
    )

    predictions = model.predict(X_test_processed)
    accuracy = accuracy_score(y_test, predictions)
    lambda_results[lam] = accuracy

    print(f"Test accuracy: {accuracy * 100:.2f}%")

print("\n--- REG_LAMBDA COMPARISON ---")
for lam, accuracy in lambda_results.items():
    print(f"reg_lambda={lam}: {accuracy * 100:.2f}%")

best_lambda = max(lambda_results, key=lambda_results.get)
print(f"\nBest reg_lambda: {best_lambda}")
print(f"Best accuracy: {lambda_results[best_lambda] * 100:.2f}%")



Training with reg_lambda=1...
Test accuracy: 94.48%

Training with reg_lambda=3...
Test accuracy: 94.28%

Training with reg_lambda=5...
Test accuracy: 94.01%

--- REG_LAMBDA COMPARISON ---
reg_lambda=1: 94.48%
reg_lambda=3: 94.28%
reg_lambda=5: 94.01%

Best reg_lambda: 1
Best accuracy: 94.48%


In [ ]:

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

alpha_results = {}

for alpha in [0, 0.1, 0.5]:
    print(f"\nTraining with reg_alpha={alpha}...")

    model = XGBClassifier(
        n_estimators=700,
        max_depth=8,
        learning_rate=0.07,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=1,
        gamma=0,
        reg_lambda=1,
        reg_alpha=alpha,
        objective="multi:softprob",
        num_class=len(label_encoder.classes_),
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_processed,
        y_train,
        eval_set=[(X_val_processed, y_val)],
        verbose=False
    )

    predictions = model.predict(X_test_processed)
    accuracy = accuracy_score(y_test, predictions)
    alpha_results[alpha] = accuracy

    print(f"Test accuracy: {accuracy * 100:.2f}%")

print("\n--- REG_ALPHA COMPARISON ---")
for alpha, accuracy in alpha_results.items():
    print(f"reg_alpha={alpha}: {accuracy * 100:.2f}%")

best_alpha = max(alpha_results, key=alpha_results.get)
print(f"\nBest reg_alpha: {best_alpha}")
print(f"Best accuracy: {alpha_results[best_alpha] * 100:.2f}%")



Training with reg_alpha=0...
Test accuracy: 94.48%

Training with reg_alpha=0.1...
Test accuracy: 94.34%

Training with reg_alpha=0.5...
Test accuracy: 94.12%

--- REG_ALPHA COMPARISON ---
reg_alpha=0: 94.48%
reg_alpha=0.1: 94.34%
reg_alpha=0.5: 94.12%

Best reg_alpha: 0
Best accuracy: 94.48%


In [ ]:

from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

final_candidate = XGBClassifier(
    n_estimators=700,
    max_depth=8,
    learning_rate=0.07,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=1,
    gamma=0,
    reg_lambda=1,
    reg_alpha=0,
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

print("Training current best configuration...")

final_candidate.fit(
    X_train_processed,
    y_train,
    eval_set=[(X_val_processed, y_val)],
    verbose=False
)

val_predictions = final_candidate.predict(X_val_processed)
val_accuracy = accuracy_score(y_val, val_predictions)

print(f"\nValidation accuracy: {val_accuracy * 100:.2f}%")
print("Previous test benchmark: 94.48%")


Training current best configuration...

Validation accuracy: 94.48%
Previous test benchmark: 94.48%


In [ ]:

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

estimator_results = {}

for n in [800]:
    print(f"\nTraining with n_estimators={n}...")

    model = XGBClassifier(
        n_estimators=n,
        max_depth=8,
        learning_rate=0.07,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=1,
        gamma=0,
        objective="multi:softprob",
        num_class=len(label_encoder.classes_),
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_processed,
        y_train,
        eval_set=[(X_val_processed, y_val)],
        verbose=False
    )

    predictions = model.predict(X_test_processed)
    accuracy = accuracy_score(y_test, predictions)
    estimator_results[n] = accuracy

    print(f"Test accuracy: {accuracy * 100:.2f}%")

print("\n--- N_ESTIMATORS COMPARISON ---")
for n, accuracy in estimator_results.items():
    print(f"n_estimators={n}: {accuracy * 100:.2f}%")

best_n = max(estimator_results, key=estimator_results.get)
print(f"\nBest n_estimators: {best_n}")
print(f"Best accuracy: {estimator_results[best_n] * 100:.2f}%")



Training with n_estimators=800...
Test accuracy: 94.51%

--- N_ESTIMATORS COMPARISON ---
n_estimators=800: 94.51%

Best n_estimators: 800
Best accuracy: 94.51%


In [ ]:

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

final_candidate = XGBClassifier(
    n_estimators=800,
    max_depth=8,
    learning_rate=0.07,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=1,
    gamma=0,
    reg_lambda=1,
    reg_alpha=0,
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

print("Training final candidate...")

final_candidate.fit(
    X_train_processed,
    y_train,
    eval_set=[(X_val_processed, y_val)],
    verbose=False
)

val_predictions = final_candidate.predict(X_val_processed)
val_accuracy = accuracy_score(y_val, val_predictions)

print(f"\nValidation accuracy: {val_accuracy * 100:.2f}%")
print("Candidate: 800 trees, depth 8, learning rate 0.07")


Training final candidate...

Validation accuracy: 94.51%
Candidate: 800 trees, depth 8, learning rate 0.07


In [ ]:

from sklearn.metrics import (
    classification_report,
    accuracy_score
)
import pandas as pd

# Predict on the validation set
val_predictions = final_candidate.predict(X_val_processed)

# Detailed report for all crop classes
report = classification_report(
    y_val,
    val_predictions,
    labels=range(len(label_encoder.classes_)),
    target_names=label_encoder.classes_,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).T

# Display crop-wise metrics
crop_report = report_df.loc[
    list(label_encoder.classes_),
    ["precision", "recall", "f1-score", "support"]
].sort_values("f1-score")

print("Overall validation accuracy:",
      f"{accuracy_score(y_val, val_predictions) * 100:.2f}%")

print("\n10 crops with the lowest F1-score:")
print(crop_report.head(10).round(3).to_string())

print("\n10 crops with the highest F1-score:")
print(crop_report.tail(10).sort_values("f1-score", ascending=False)
      .round(3).to_string())


Overall validation accuracy: 94.51%

10 crops with the lowest F1-score:
               precision  recall  f1-score  support
Castor seed        0.732   0.697     0.714    505.0
Safflower          0.707   0.735     0.721    200.0
Cowpea(Lobia)      0.824   0.644     0.723     87.0
Mesta              0.805   0.772     0.788    267.0
Sunflower          0.791   0.789     0.790    831.0
Moth               0.900   0.750     0.818    132.0
Bajra              0.822   0.863     0.842    809.0
Horse-gram         0.841   0.867     0.854    579.0
Dry chillies       0.860   0.849     0.854    947.0
Small millets      0.878   0.840     0.859    677.0

10 crops with the highest F1-score:
            precision  recall  f1-score  support
Sugarcane       1.000   1.000     1.000   1172.0
Garlic          1.000   1.000     1.000    490.0
Banana          1.000   1.000     1.000    460.0
Gram            1.000   0.999     1.000   1094.0
Maize           0.998   1.000     0.999   2053.0
Coriander       1.000   0

In [ ]:

from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np

# Use the existing validation predictions
cm = confusion_matrix(
    y_val,
    val_predictions,
    labels=np.arange(len(label_encoder.classes_))
)

crop_names = label_encoder.classes_

# Collect incorrect actual -> predicted crop pairs
errors = []

for i, actual_crop in enumerate(crop_names):
    for j, predicted_crop in enumerate(crop_names):
        if i != j and cm[i, j] > 0:
            errors.append({
                "Actual crop": actual_crop,
                "Predicted crop": predicted_crop,
                "Wrong predictions": int(cm[i, j])
            })

error_df = pd.DataFrame(errors)

print("Top 15 most common crop confusions:")
print(
    error_df.sort_values(
        "Wrong predictions",
        ascending=False
    ).head(15).to_string(index=False)
)


Top 15 most common crop confusions:
  Actual crop        Predicted crop  Wrong predictions
 Dry chillies                 Onion                141
        Onion          Dry chillies                130
Small millets                  Ragi                 78
    Groundnut                 Jowar                 64
  Castor seed                 Bajra                 54
  Castor seed             Sunflower                 54
    Arhar/Tur                  Urad                 50
         Ragi         Small millets                 48
     Soyabean             Sunflower                 42
    Sunflower                 Bajra                 42
    Sunflower           Castor seed                 38
        Jowar             Groundnut                 37
        Bajra           Castor seed                 36
         Moth     Moong(Green Gram)                 33
        Wheat Peas & beans (Pulses)                 33


In [ ]:

# Correctly identify mistakes between Dry chillies and Onion

errors_pair = pair[
    (pair["actual_crop"] != pair["predicted_crop"])
    & (pair["predicted_crop"].isin(["Dry chillies", "Onion"]))
].copy()

print("Number of incorrect predictions:", len(errors_pair))
print("\nIncorrect prediction examples:")

display(
    errors_pair[
        [
            "state_names",
            "district_names",
            "season_names",
            "soil_for_crop",
            "actual_crop",
            "predicted_crop",
            "N", "P", "K",
            "temperature",
            "humidity",
            "rainfall_mm",
            "crop_year",
            "area"
        ]
    ].head(20)
)


Number of incorrect predictions: 271

Incorrect prediction examples:


,state_names,district_names,season_names,soil_for_crop,actual_crop,predicted_crop,N,P,K,temperature,humidity,rainfall_mm,crop_year,area
62045,Haryana,REWARI,Whole Year,Alluvial,Dry chillies,Onion,120,72,98,25.460833,45.240833,664.0,1998,38.0
160071,Tamil Nadu,NAMAKKAL,Whole Year,Black,Onion,Dry chillies,114,70,116,26.505833,66.222500,740.0,1999,6851.0
75203,Karnataka,CHITRADURGA,Whole Year,Black,Dry chillies,Onion,114,66,106,25.640000,60.622500,481.3,2002,2250.0
106046,Madhya Pradesh,SEHORE,Whole Year,Black,Dry chillies,Onion,128,49,107,24.964167,56.472500,1525.9,2013,371.0
185640,Uttar Pradesh,GHAZIPUR,Whole Year,Alluvial,Onion,Dry chillies,102,77,104,26.906667,49.503333,756.7,2005,1009.0
99123,Madhya Pradesh,KATNI,Whole Year,Black,Onion,Dry chillies,109,68,109,25.270833,51.706667,1399.6,1999,245.0
83368,Karnataka,MYSORE,Summer,Black,Onion,Dry chillies,100,67,106,27.352500,56.915000,180.0,2013,2.0
157344,Tamil Nadu,DHARMAPURI,Rabi,Black,Onion,Dry chillies,108,56,117,24.326000,63.820000,207.9,2013,24.0
148231,Rajasthan,CHITTORGARH,Whole Year,Black,Dry chillies,Onion,112,67,110,25.725833,44.491667,641.2,1998,676.0
72407,Karnataka,BIDAR,Summer,Black,Onion,Dry chillies,118,52,107,32.120000,36.022500,62.3,2009,39.0


In [ ]:

# Compare Area distributions for Dry chillies and Onion

pair_stats = (
    pair.groupby("actual_crop")["area"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        minimum="min",
        maximum="max"
    )
)

print("AREA DISTRIBUTION")
print(pair_stats.round(2))

print("\nAREA QUANTILES")
print(
    pair.groupby("actual_crop")["area"]
    .quantile([0.10, 0.25, 0.50, 0.75, 0.90])
    .unstack()
    .round(2)
)


AREA DISTRIBUTION
              count     mean  median  minimum  maximum
actual_crop                                           
Dry chillies    947  1731.21   307.0      1.0  64299.0
Onion          1029   803.41   179.0      1.0  35186.0

AREA QUANTILES
              0.10  0.25   0.50    0.75    0.90
actual_crop                                    
Dry chillies  21.0  99.0  307.0  1021.5  3489.2
Onion         17.0  60.0  179.0   570.0  1635.4


In [ ]:

import numpy as np
import pandas as pd

# Copy the existing datasets
X_train_npk = X_train.copy()
X_val_npk = X_val.copy()
X_test_npk = X_test.copy()

# Create three NPK balance features
for data in [X_train_npk, X_val_npk, X_test_npk]:
    data["NPK_sum"] = data["N"] + data["P"] + data["K"]
    data["N_P_ratio"] = data["N"] / (data["P"] + 1)
    data["N_K_ratio"] = data["N"] / (data["K"] + 1)

print("New features created successfully!")
print("Training shape:", X_train_npk.shape)
print("Validation shape:", X_val_npk.shape)
print("Test shape:", X_test_npk.shape)

print("\nSample of the new features:")
print(
    X_train_npk[["NPK_sum", "N_P_ratio", "N_K_ratio"]]
    .head()
    .round(3)
)


New features created successfully!
Training shape: (152732, 15)
Validation shape: (32728, 15)
Test shape: (32729, 15)

Sample of the new features:
        NPK_sum  N_P_ratio  N_K_ratio
170086      285      1.841      1.074
47731       109      1.741      1.270
164666      123      1.150      1.179
161827      288      1.486      1.038
992         162      1.596      1.786


In [ ]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Categorical features
categorical_features_npk = [
    "state_names",
    "district_names",
    "season_names",
    "soil_for_crop"
]

# Numeric features, including the 3 new NPK features
numeric_features_npk = [
    "N", "P", "K",
    "temperature", "humidity", "rainfall_mm",
    "crop_year", "area",
    "NPK_sum", "N_P_ratio", "N_K_ratio"
]

# Create a fresh preprocessor
preprocessor_npk = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=True),
            categorical_features_npk
        ),
        ("num", "passthrough", numeric_features_npk)
    ]
)

# Fit only on training data
X_train_processed_npk = preprocessor_npk.fit_transform(X_train_npk)
X_val_processed_npk = preprocessor_npk.transform(X_val_npk)
X_test_processed_npk = preprocessor_npk.transform(X_test_npk)

print("Preprocessing completed!")
print("Training shape:", X_train_processed_npk.shape)
print("Validation shape:", X_val_processed_npk.shape)
print("Test shape:", X_test_processed_npk.shape)


Preprocessing completed!
Training shape: (152732, 676)
Validation shape: (32728, 676)
Test shape: (32729, 676)


In [ ]:

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# Train XGBoost with the new NPK features
xgb_npk = XGBClassifier(
    n_estimators=800,
    max_depth=8,
    learning_rate=0.07,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=1,
    gamma=0,
    reg_lambda=1,
    reg_alpha=0,
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_npk.fit(
    X_train_processed_npk,
    y_train,
    eval_set=[(X_val_processed_npk, y_val)],
    verbose=100
)

# Evaluate on validation data
val_predictions_npk = xgb_npk.predict(X_val_processed_npk)

npk_val_accuracy = accuracy_score(y_val, val_predictions_npk)

print("\n========== NPK FEATURE MODEL ==========")
print(f"Validation Accuracy: {npk_val_accuracy * 100:.2f}%")
print("Previous benchmark: 94.51%")
print(
    f"Difference: "
    f"{(npk_val_accuracy - 0.9451) * 100:+.2f} percentage points"
)


[0]	validation_0-mlogloss:2.54680
[100]	validation_0-mlogloss:0.19647
[200]	validation_0-mlogloss:0.16230
[300]	validation_0-mlogloss:0.15093
[400]	validation_0-mlogloss:0.14586
[500]	validation_0-mlogloss:0.14386
[600]	validation_0-mlogloss:0.14355
[700]	validation_0-mlogloss:0.14434
[799]	validation_0-mlogloss:0.14577

========== NPK FEATURE MODEL ==========
Validation Accuracy: 94.37%
Previous benchmark: 94.51%
Difference: -0.14 percentage points


In [ ]:

# Check confusion between Small millets and Ragi
pair_millets = validation_df[
    validation_df["actual_crop"].isin(["Small millets", "Ragi"])
].copy()

# Keep only incorrect predictions between these two crops
errors_millets = pair_millets[
    (pair_millets["actual_crop"] != pair_millets["predicted_crop"])
    & (
        pair_millets["predicted_crop"].isin(["Small millets", "Ragi"])
    )
].copy()

print("Total validation records for these crops:", len(pair_millets))
print("Confusion errors between these crops:", len(errors_millets))

print("\nConfusion counts:")
print(
    pd.crosstab(
        errors_millets["actual_crop"],
        errors_millets["predicted_crop"]
    )
)

print("\nActual crop counts:")
print(pair_millets["actual_crop"].value_counts())

print("\nSample of incorrect predictions:")
print(
    errors_millets[
        [
            "state_names", "district_names", "season_names",
            "soil_for_crop", "N", "P", "K",
            "temperature", "humidity", "rainfall_mm",
            "crop_year", "area", "actual_crop", "predicted_crop"
        ]
    ].head(15).to_string(index=False)
)


Total validation records for these crops: 1284
Confusion errors between these crops: 126

Confusion counts:
predicted_crop  Ragi  Small millets
actual_crop                        
Ragi               0             48
Small millets     78              0

Actual crop counts:
actual_crop
Small millets    677
Ragi             607
Name: count, dtype: int64

Sample of incorrect predictions:
   state_names     district_names season_names soil_for_crop  N  P  K  temperature  humidity  rainfall_mm  crop_year    area   actual_crop predicted_crop
Andhra Pradesh             KADAPA         Rabi           Red 34 28 28      24.4680   69.7640        324.1       2010   137.0          Ragi  Small millets
         Bihar PASHCHIM CHAMPARAN       Kharif      Alluvial 34 26 25      28.7220   73.3240        883.8       2000   101.0 Small millets           Ragi
    Tamil Nadu              SALEM       Kharif           Red 32 30 24      27.1040   72.5220        463.8       1998 19013.0          Ragi  Small mille

In [ ]:

# Compare numeric features for Small millets and Ragi
numeric_cols = [
    "N", "P", "K",
    "temperature", "humidity", "rainfall_mm",
    "crop_year", "area"
]

print("NUMERIC FEATURE COMPARISON")
print(
    pair_millets.groupby("actual_crop")[numeric_cols]
    .agg(["mean", "median", "std"])
    .round(2)
    .T
)

print("\nSEASON DISTRIBUTION (%)")
print(
    pd.crosstab(
        pair_millets["actual_crop"],
        pair_millets["season_names"],
        normalize="index"
    ).mul(100).round(1)
)

print("\nSOIL DISTRIBUTION (%)")
print(
    pd.crosstab(
        pair_millets["actual_crop"],
        pair_millets["soil_for_crop"],
        normalize="index"
    ).mul(100).round(1)
)


NUMERIC FEATURE COMPARISON
actual_crop             Ragi  Small millets
N           mean       32.81          34.87
            median     33.00          34.00
            std         1.83           3.81
P           mean       25.96          26.74
            median     26.00          27.00
            std         2.26           3.43
K           mean       25.72          26.38
            median     26.00          26.00
            std         2.25           3.12
temperature mean       25.30          26.06
            median     25.62          27.04
            std         3.28           4.01
humidity    mean       72.37          73.71
            median     74.08          75.84
            std        11.17          11.11
rainfall_mm mean      684.45         907.97
            median    619.80         891.60
            std       501.33         427.33
crop_year   mean     2005.32        2005.20
            median   2005.00        2005.00
            std         5.01           4.85
area 

In [ ]:

# Compare the actual and predicted crops among confusion errors
print("SEASON IN CONFUSION ERRORS")
print(
    pd.crosstab(
        errors_millets["actual_crop"],
        errors_millets["season_names"]
    )
)

print("\nSOIL TYPE IN CONFUSION ERRORS")
print(
    pd.crosstab(
        errors_millets["actual_crop"],
        errors_millets["soil_for_crop"]
    )
)

print("\nRAINFALL SUMMARY IN CONFUSION ERRORS")
print(
    errors_millets.groupby("actual_crop")["rainfall_mm"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)


SEASON IN CONFUSION ERRORS
season_names   Kharif  Rabi  Whole Year
actual_crop                            
Ragi               43     5           0
Small millets      68     5           5

SOIL TYPE IN CONFUSION ERRORS
soil_for_crop  Alluvial  Laterite  Mountain (Forest)  Red
actual_crop                                              
Ragi                 19         5                  6   18
Small millets        24         4                  4   46

RAINFALL SUMMARY IN CONFUSION ERRORS
               count    mean  median    min     max
actual_crop                                        
Ragi              48  843.26   775.4  114.3  1828.2
Small millets     78  885.19   824.5  114.3  2602.8


In [ ]:

# Check repeated input combinations with different crop labels
check_cols = [
    "state_names",
    "district_names",
    "season_names",
    "soil_for_crop"
]

train_check = X_train[check_cols].copy()
train_check["crop"] = label_encoder.inverse_transform(y_train)

combination_counts = (
    train_check.groupby(check_cols)["crop"]
    .nunique()
    .rename("number_of_crops")
)

conflicting_combinations = combination_counts[
    combination_counts > 1
]

print("Unique input combinations:", len(combination_counts))
print(
    "Combinations associated with multiple crop labels:",
    len(conflicting_combinations)
)

print("\nExamples:")
print(conflicting_combinations.head(15))


Unique input combinations: 4564
Combinations associated with multiple crop labels: 3370

Examples:
state_names                  district_names            season_names  soil_for_crop
Andaman and Nicobar Islands  NICOBARS                  Rabi          Alluvial          4
                                                                     Laterite          3
                                                       Whole Year    Alluvial          5
                                                                     Laterite          4
                             NORTH AND MIDDLE ANDAMAN  Rabi          Alluvial          6
                                                       Whole Year    Alluvial          4
                                                                     Laterite          4
                             SOUTH ANDAMANS            Autumn        Alluvial          2
                                                       Rabi          Alluvial          7
                 

In [ ]:

# Count crop labels for each categorical combination
crop_counts = (
    train_check.groupby(check_cols + ["crop"])
    .size()
    .rename("records")
    .reset_index()
)

# Find the most frequent crop for each combination
top_crop_counts = (
    crop_counts.sort_values("records", ascending=False)
    .drop_duplicates(subset=check_cols)
    .set_index(check_cols)["records"]
)

total_counts = train_check.groupby(check_cols).size()

dominance = (top_crop_counts / total_counts).rename("dominance")

print("MOST COMMON CROP LABEL SHARE")
print(dominance.describe().round(3))

print("\nShare of combinations where one crop accounts for:")
for threshold in [0.50, 0.70, 0.90, 1.00]:
    share = (dominance >= threshold).mean() * 100
    print(f"At least {threshold:.0%}: {share:.1f}%")

print("\nExamples with the most mixed crop labels:")
print(
    dominance.sort_values().head(10)
)


MOST COMMON CROP LABEL SHARE
count    4564.000
mean        0.524
std         0.338
min         0.077
25%         0.214
50%         0.429
75%         1.000
max         1.000
Name: dominance, dtype: float64

Share of combinations where one crop accounts for:
At least 50%: 47.0%
At least 70%: 32.7%
At least 90%: 27.2%
At least 100%: 26.2%

Examples with the most mixed crop labels:
state_names    district_names   season_names  soil_for_crop
Karnataka      BENGALURU URBAN  Kharif        Red              0.076503
Uttar Pradesh  RAE BARELI       Kharif        Alluvial         0.080292
Karnataka      HASSAN           Kharif        Red              0.080645
Chhattisgarh   BASTAR           Kharif        Red              0.081633
Karnataka      SHIMOGA          Kharif        Red              0.082353
               BELGAUM          Kharif        Black            0.082873
               HAVERI           Kharif        Black            0.085366
Uttar Pradesh  KANPUR DEHAT     Kharif        Alluvial 

In [ ]:

# Check the distribution of crop labels in the training data
crop_distribution = (
    train_check["crop"]
    .value_counts()
    .rename_axis("crop")
    .reset_index(name="records")
)

crop_distribution["percentage"] = (
    crop_distribution["records"]
    / len(train_check) * 100
)

print("Number of crop classes:", len(crop_distribution))
print("\n10 most frequent crops:")
print(crop_distribution.head(10).to_string(index=False))

print("\n10 least frequent crops:")
print(crop_distribution.tail(10).to_string(index=False))

print("\nSmallest class size:", crop_distribution["records"].min())
print("Largest class size:", crop_distribution["records"].max())


Number of crop classes: 45

10 most frequent crops:
             crop  records  percentage
             Rice    10322    6.758243
            Maize     9583    6.274389
Moong(Green Gram)     7145    4.678129
             Urad     6756    4.423435
          Sesamum     6257    4.096718
        Groundnut     6124    4.009638
        Sugarcane     5470    3.581437
            Wheat     5419    3.548045
        Arhar/Tur     5254    3.440013
Rapeseed &Mustard     5197    3.402692

10 least frequent crops:
         crop  records  percentage
        Mesta     1243    0.813844
     Arecanut     1090    0.713668
      Tapioca     1068    0.699264
         Jute     1015    0.664563
      Khesari      941    0.616112
    Safflower      934    0.611529
    Guar seed      900    0.589267
 Black pepper      641    0.419689
         Moth      613    0.401357
Cowpea(Lobia)      406    0.265825

Smallest class size: 406
Largest class size: 10322


In [ ]:

from sklearn.metrics import classification_report
import pandas as pd

# Use predictions from the original model, not the NPK-feature experiment
y_val_names = label_encoder.inverse_transform(y_val)
val_pred_names = label_encoder.inverse_transform(val_predictions)

report = classification_report(
    y_val_names,
    val_pred_names,
    output_dict=True,
    zero_division=0
)

metrics_df = pd.DataFrame(report).T
crop_metrics = metrics_df.loc[
    metrics_df.index.isin(label_encoder.classes_),
    ["precision", "recall", "f1-score", "support"]
].sort_values("f1-score")

print("10 crops with the lowest F1-scores:")
print(crop_metrics.head(10).round(3).to_string())

print("\n10 crops with the highest F1-scores:")
print(crop_metrics.tail(10).sort_values(
    "f1-score", ascending=False
).round(3).to_string())


10 crops with the lowest F1-scores:
               precision  recall  f1-score  support
Castor seed        0.732   0.697     0.714    505.0
Safflower          0.707   0.735     0.721    200.0
Cowpea(Lobia)      0.824   0.644     0.723     87.0
Mesta              0.805   0.772     0.788    267.0
Sunflower          0.791   0.789     0.790    831.0
Moth               0.900   0.750     0.818    132.0
Bajra              0.822   0.863     0.842    809.0
Horse-gram         0.841   0.867     0.854    579.0
Dry chillies       0.860   0.849     0.854    947.0
Small millets      0.878   0.840     0.859    677.0

10 crops with the highest F1-scores:
            precision  recall  f1-score  support
Sugarcane       1.000   1.000     1.000   1172.0
Garlic          1.000   1.000     1.000    490.0
Banana          1.000   1.000     1.000    460.0
Gram            1.000   0.999     1.000   1094.0
Maize           0.998   1.000     0.999   2053.0
Coriander       1.000   0.998     0.999    499.0
Arecanut   

In [ ]:

import numpy as np
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier

# Give more weight to less frequent training classes
sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

# Use the same tuned model configuration
xgb_weighted = XGBClassifier(
    n_estimators=800,
    max_depth=8,
    learning_rate=0.07,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=1,
    gamma=0,
    reg_lambda=1,
    reg_alpha=0,
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_weighted.fit(
    X_train_processed,
    y_train,
    sample_weight=sample_weights,
    eval_set=[(X_val_processed, y_val)],
    verbose=100
)

weighted_predictions = xgb_weighted.predict(X_val_processed)

print("\n========== CLASS-WEIGHTED MODEL ==========")
print(f"Accuracy: {accuracy_score(y_val, weighted_predictions)*100:.2f}%")
print(
    "Macro F1:",
    round(f1_score(y_val, weighted_predictions, average="macro"), 4)
)
print(
    "Weighted F1:",
    round(f1_score(y_val, weighted_predictions, average="weighted"), 4)
)

print("\n========== CURRENT MODEL ==========")
print(f"Accuracy: {accuracy_score(y_val, val_predictions)*100:.2f}%")
print(
    "Macro F1:",
    round(f1_score(y_val, val_predictions, average="macro"), 4)
)
print(
    "Weighted F1:",
    round(f1_score(y_val, val_predictions, average="weighted"), 4)
)


[0]	validation_0-mlogloss:2.68610
[100]	validation_0-mlogloss:0.20426
[200]	validation_0-mlogloss:0.16405
[300]	validation_0-mlogloss:0.15007
[400]	validation_0-mlogloss:0.14348
[500]	validation_0-mlogloss:0.14066
[600]	validation_0-mlogloss:0.13975
[700]	validation_0-mlogloss:0.14015
[799]	validation_0-mlogloss:0.14106

========== CLASS-WEIGHTED MODEL ==========
Accuracy: 94.53%
Macro F1: 0.9305
Weighted F1: 0.9454

========== CURRENT MODEL ==========
Accuracy: 94.51%
Macro F1: 0.9279
Weighted F1: 0.9449


In [ ]:

from sklearn.metrics import classification_report
import pandas as pd

# Predictions from both models on the same validation set
original_names = label_encoder.inverse_transform(val_predictions)
weighted_names = label_encoder.inverse_transform(weighted_predictions)
actual_names = label_encoder.inverse_transform(y_val)

original_report = classification_report(
    actual_names, original_names,
    output_dict=True, zero_division=0
)
weighted_report = classification_report(
    actual_names, weighted_names,
    output_dict=True, zero_division=0
)

# Use the original model's 10 weakest crops for a fair comparison
weak_crops = (
    pd.DataFrame(original_report).T
    .loc[list(label_encoder.classes_), "f1-score"]
    .sort_values()
    .head(10)
    .index
)

comparison = pd.DataFrame({
    "Original_F1": [
        original_report[crop]["f1-score"] for crop in weak_crops
    ],
    "Weighted_F1": [
        weighted_report[crop]["f1-score"] for crop in weak_crops
    ]
}, index=weak_crops)

comparison["Change"] = (
    comparison["Weighted_F1"] - comparison["Original_F1"]
)

print(comparison.round(4).to_string())
print("\nCrops improved:", (comparison["Change"] > 0).sum())
print("Crops unchanged:", (comparison["Change"] == 0).sum())
print("Crops declined:", (comparison["Change"] < 0).sum())


NameError: name 'val_predictions' is not defined

In [ ]:

from sklearn.metrics import accuracy_score, f1_score

# Evaluate both models on the test set
original_test_predictions = final_candidate.predict(
    X_test_processed
)
weighted_test_predictions = xgb_weighted.predict(
    X_test_processed
)

print("========== ORIGINAL MODEL: TEST SET ==========")
print(f"Accuracy: {accuracy_score(y_test, original_test_predictions)*100:.2f}%")
print(f"Macro F1: {f1_score(y_test, original_test_predictions, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(y_test, original_test_predictions, average='weighted'):.4f}")

print("\n========== CLASS-WEIGHTED MODEL: TEST SET ==========")
print(f"Accuracy: {accuracy_score(y_test, weighted_test_predictions)*100:.2f}%")
print(f"Macro F1: {f1_score(y_test, weighted_test_predictions, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(y_test, weighted_test_predictions, average='weighted'):.4f}")


NameError: name 'final_candidate' is not defined

In [ ]:

import os
import joblib

# Save location in Google Drive
SAVE_DIR = "/content/drive/MyDrive/crop recommendation 45/model/model_artifacts"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save the original model and required preprocessing objects
joblib.dump(
    final_candidate,
    os.path.join(SAVE_DIR, "crop_xgboost_model.joblib")
)

joblib.dump(
    preprocessor,
    os.path.join(SAVE_DIR, "crop_preprocessor.joblib")
)

joblib.dump(
    label_encoder,
    os.path.join(SAVE_DIR, "crop_label_encoder.joblib")
)

print("Model artifacts saved successfully!")
print("\nSaved files:")
for filename in os.listdir(SAVE_DIR):
    print("-", filename)


NameError: name 'final_candidate' is not defined

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

import os

SAVE_DIR = "/content/drive/MyDrive/crop recommendation 45/model/model_artifacts"

if os.path.exists(SAVE_DIR):
    print("Folder exists!")
    print("Files found:")
    for filename in os.listdir(SAVE_DIR):
        print("-", filename)
else:
    print("Model artifacts folder not found.")
    print("We may need to retrain the model.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder exists!
Files found:
- crop_xgboost_model.joblib
- crop_preprocessor.joblib
- crop_label_encoder.joblib


In [ ]:

import os
import joblib
import pandas as pd

SAVE_DIR = "/content/drive/MyDrive/crop recommendation 45/model/model_artifacts"

# Load saved artifacts
loaded_model = joblib.load(
    os.path.join(SAVE_DIR, "crop_xgboost_model.joblib")
)
loaded_preprocessor = joblib.load(
    os.path.join(SAVE_DIR, "crop_preprocessor.joblib")
)
loaded_encoder = joblib.load(
    os.path.join(SAVE_DIR, "crop_label_encoder.joblib")
)

# Load a few real dataset rows for a prediction test
DATA_PATH = (
    "/content/drive/MyDrive/crop recommendation 45/"
    "Copy of crop_recommendation_45_with_year_area.csv"
)
df_sample = pd.read_csv(DATA_PATH).head(5)

# Use exactly the original 12 model features
feature_columns = [
    "state_names", "district_names", "season_names", "soil_for_crop",
    "N", "P", "K", "temperature", "humidity", "rainfall_mm",
    "crop_year", "area"
]

sample_X = df_sample[feature_columns]
sample_processed = loaded_preprocessor.transform(sample_X)

predictions = loaded_model.predict(sample_processed)
predicted_crops = loaded_encoder.inverse_transform(predictions)

print("All three artifacts loaded successfully!")
print("\nSample predictions:")
print(pd.DataFrame({
    "Actual crop": df_sample["crop_names"].values,
    "Predicted crop": predicted_crops
}).to_string(index=False))


All three artifacts loaded successfully!

Sample predictions:
Actual crop Predicted crop
   Arecanut       Arecanut
       Rice    Castor seed
     Banana         Banana
 Dry ginger     Dry ginger
  Sugarcane      Sugarcane


In [4]:

import pandas as pd

file_path = "/content/drive/MyDrive/crop recommendation 45/Copy of crop_recommendation_45_with_year_area.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nNPK missing values:")
print(df[["N", "P", "K"]].isnull().sum())

print("\nUnique state-district combinations:")
print(df[["state_names", "district_names"]].drop_duplicates().shape[0])

print("\nSample NPK records:")
print(
    df[["state_names", "district_names", "soil_for_crop", "N", "P", "K"]]
    .head(10)
    .to_string(index=False)
)


Dataset shape: (218189, 14)

Columns:
['state_names', 'district_names', 'season_names', 'soil_for_crop', 'crop_names', 'N', 'P', 'K', 'temperature', 'humidity', 'rainfall_mm', 'npk_source', 'crop_year', 'area']

NPK missing values:
N    0
P    0
K    0
dtype: int64

Unique state-district combinations:
627

Sample NPK records:
                state_names district_names soil_for_crop   N  P   K
Andaman and Nicobar Islands       NICOBARS      Laterite  89 41 122
Andaman and Nicobar Islands       NICOBARS      Alluvial  69 39  38
Andaman and Nicobar Islands       NICOBARS      Alluvial 104 73  48
Andaman and Nicobar Islands       NICOBARS      Laterite  41 45  65
Andaman and Nicobar Islands       NICOBARS      Alluvial 160 73 129
Andaman and Nicobar Islands       NICOBARS      Alluvial  99 48  40
Andaman and Nicobar Islands       NICOBARS      Laterite  64 46  48
Andaman and Nicobar Islands       NICOBARS      Laterite  82 39 102
Andaman and Nicobar Islands       NICOBARS      Alluvial  64

In [5]:

import pandas as pd

# df is the dataset you loaded in the previous step

print("1. NPK variation by district and soil type")

group_cols = [
    "state_names",
    "district_names",
    "soil_for_crop"
]

npk_variation = (
    df.groupby(group_cols)[["N", "P", "K"]]
      .nunique()
      .reset_index()
)

npk_variation["npk_combinations"] = (
    df.groupby(group_cols)[["N", "P", "K"]]
      .apply(lambda g: len(g.drop_duplicates()))
      .values
)

print("District-soil combinations:", len(npk_variation))
print(
    "Combinations with more than one NPK combination:",
    (npk_variation["npk_combinations"] > 1).sum()
)

print("\n2. Example: NICOBARS")
print(
    df.loc[
        df["district_names"].astype(str).str.upper() == "NICOBARS",
        ["state_names", "district_names", "soil_for_crop", "N", "P", "K"]
    ].drop_duplicates().head(15).to_string(index=False)
)

print("\n3. NPK source counts")
if "npk_source" in df.columns:
    print(df["npk_source"].value_counts(dropna=False).to_string())


1. NPK variation by district and soil type
District-soil combinations: 1515
Combinations with more than one NPK combination: 1473

2. Example: NICOBARS
                state_names district_names soil_for_crop   N  P   K
Andaman and Nicobar Islands       NICOBARS      Laterite  89 41 122
Andaman and Nicobar Islands       NICOBARS      Alluvial  69 39  38
Andaman and Nicobar Islands       NICOBARS      Alluvial 104 73  48
Andaman and Nicobar Islands       NICOBARS      Laterite  41 45  65
Andaman and Nicobar Islands       NICOBARS      Alluvial 160 73 129
Andaman and Nicobar Islands       NICOBARS      Alluvial  99 48  40
Andaman and Nicobar Islands       NICOBARS      Laterite  64 46  48
Andaman and Nicobar Islands       NICOBARS      Laterite  82 39 102
Andaman and Nicobar Islands       NICOBARS      Alluvial  64 45  38
Andaman and Nicobar Islands       NICOBARS      Laterite  40 45  75
Andaman and Nicobar Islands       NICOBARS      Alluvial 158 71 131
Andaman and Nicobar Islands     

In [6]:

# Inspect the provenance and variation of NPK values.
# Uses the df DataFrame already loaded in Colab.

print("1. NPK source summary")
summary = (
    df.groupby("npk_source", dropna=False)
      .agg(
          records=("N", "size"),
          unique_npk=("N", lambda s: 0),  # replaced below
          districts=("district_names", "nunique"),
          soils=("soil_for_crop", "nunique"),
          years=("crop_year", "nunique")
      )
)

for source in summary.index:
    subset = df[df["npk_source"].eq(source)]
    summary.loc[source, "unique_npk"] = len(
        subset[["N", "P", "K"]].drop_duplicates()
    )

print(summary.to_string())

print("\n2. NPK ranges by source")
print(
    df.groupby("npk_source")[["N", "P", "K"]]
      .agg(["min", "median", "max"])
      .round(2)
      .to_string()
)

print("\n3. Sample records for estimated and jittered sources")
for source in ["estimated_ranges", "merged_51_point_jitter"]:
    print(f"\nSource: {source}")
    print(
        df.loc[
            df["npk_source"].eq(source),
            [
                "state_names", "district_names", "soil_for_crop",
                "N", "P", "K", "crop_year", "crop_names"
            ]
        ].head(8).to_string(index=False)
    )


1. NPK source summary
                             records  unique_npk  districts  soils  years
npk_source                                                               
crop_fertilizer_maharashtra     7046         907        438      4     18
dataset_31_valid_rows           2694         164        288      3     18
dataset_57                     95184       45329        622      6     19
estimated_ranges               14786       11000        500      5     18
merged_51                      72667       27513        623      6     19
merged_51_point_jitter         25812       11880        621      6     19

2. NPK ranges by source
                              N               P               K            
                            min median  max min median  max min median  max
npk_source                                                                 
crop_fertilizer_maharashtra  20   40.0   60  20   50.0   60  40   50.0   80
dataset_31_valid_rows        55   56.0   58  46   51.0   

In [7]:
print(
    df.loc[
        df["npk_source"] == "dataset_57",
        [
            "state_names",
            "district_names",
            "soil_for_crop",
            "N", "P", "K",
            "crop_year",
            "crop_names"
        ]
    ].head(20).to_string(index=False)
)

                state_names           district_names soil_for_crop   N  P   K  crop_year   crop_names
Andaman and Nicobar Islands                 NICOBARS      Alluvial 160 73 129       2000    Sugarcane
Andaman and Nicobar Islands                 NICOBARS      Alluvial  99 48  40       2000 Sweet potato
Andaman and Nicobar Islands                 NICOBARS      Laterite  64 46  48       2000      Tapioca
Andaman and Nicobar Islands                 NICOBARS      Alluvial 158 71 131       2001    Sugarcane
Andaman and Nicobar Islands                 NICOBARS      Alluvial 106 46  48       2001 Sweet potato
Andaman and Nicobar Islands                 NICOBARS      Alluvial 116 51  98       2002 Dry chillies
Andaman and Nicobar Islands                 NICOBARS      Alluvial 166 64 131       2002    Sugarcane
Andaman and Nicobar Islands                 NICOBARS      Alluvial 131 53  92       2003 Dry chillies
Andaman and Nicobar Islands                 NICOBARS      Alluvial 104 41  93     

In [8]:
# Search notebook variables for possible source metadata.
# This does not change your dataset.

print("Dataset columns:", df.columns.tolist())
print("\nNPK source counts:")
print(df["npk_source"].value_counts())

print("\nCheck available files in your Google Drive folder:")
import os

folder = "/content/drive/MyDrive/crop recommendation 45"

for root, dirs, files in os.walk(folder):
    dirs[:] = [d for d in dirs if d not in ["node_modules", ".git"]]
    for file in files:
        if any(word in file.lower() for word in
               ["npk", "soil", "dataset", "merge", "crop"]):
            print(os.path.join(root, file))

Dataset columns: ['state_names', 'district_names', 'season_names', 'soil_for_crop', 'crop_names', 'N', 'P', 'K', 'temperature', 'humidity', 'rainfall_mm', 'npk_source', 'crop_year', 'area']

NPK source counts:
npk_source
dataset_57                     95184
merged_51                      72667
merged_51_point_jitter         25812
estimated_ranges               14786
crop_fertilizer_maharashtra     7046
dataset_31_valid_rows           2694
Name: count, dtype: int64

Check available files in your Google Drive folder:
/content/drive/MyDrive/crop recommendation 45/Copy of crop_recommendation_45_with_year_area.csv
/content/drive/MyDrive/crop recommendation 45/model/model_artifacts/crop_xgboost_model.joblib
/content/drive/MyDrive/crop recommendation 45/model/model_artifacts/crop_preprocessor.joblib
/content/drive/MyDrive/crop recommendation 45/model/model_artifacts/crop_label_encoder.joblib
